# Ejercicio 3 — Transición de $3^2$ a CCD (R)

**Objetivo.** Partir del $3^2$, aumentarlo con puntos axiales y centros (CCD) y comparar
las estimaciones del modelo de segundo orden.

**Factores:** Tiempo de curado ($A$: 20/30/40 min) y Temperatura ($B$: 140/160/180 °C)
**Respuesta:** Adherencia de la pintura (N/mm²)

In [ ]:
library(dplyr)
library(ggplot2)
library(rsm)

df32 <- read.csv('../../datos/calidad-pintura-3k.csv')
cat('Diseño 3^2:\n'); print(df32[,c('x1','x2','adherencia')])

## 1. Modelo sobre el $3^2$

In [ ]:
modelo_32 <- lm(adherencia ~ x1 + x2 + I(x1^2) + I(x2^2) + x1:x2, data=df32)
print(summary(modelo_32))

## 2. Construir el CCD

In [ ]:
alpha <- sqrt(2)
# Puntos axiales coherentes con la superficie del 3² (ruido real ±0.3 N/mm²)
axiales <- data.frame(x1=c(alpha,-alpha,0,0), x2=c(0,0,alpha,-alpha),
                      adherencia=c(61.4, 44.3, 58.6, 47.9))
centros_extra <- data.frame(x1=c(0,0,0), x2=c(0,0,0), adherencia=c(78.5,80.1,79.0))

df_ccd <- bind_rows(df32[,c('x1','x2','adherencia')], axiales, centros_extra)
cat(sprintf('CCD completo: %d corridas\n', nrow(df_ccd)))
print(df_ccd)

## 3. Modelo sobre el CCD y punto óptimo

In [ ]:
modelo_ccd <- lm(adherencia ~ x1 + x2 + I(x1^2) + I(x2^2) + x1:x2, data=df_ccd)
print(summary(modelo_ccd))

modelo_rsm <- rsm(adherencia ~ SO(x1, x2), data=df_ccd)
cat('\nPunto estacionario:\n')
print(canonical(modelo_rsm))

## 4. Comparación de coeficientes

In [ ]:
terminos <- c('(Intercept)','x1','x2','I(x1^2)','I(x2^2)','x1:x2')
comp <- data.frame(
  Termino = c('β₀','β₁','β₂','β₁₁','β₂₂','β₁₂'),
  Est_32  = round(coef(modelo_32)[terminos], 3),
  SE_32   = round(sqrt(diag(vcov(modelo_32)))[terminos], 3),
  Est_CCD = round(coef(modelo_ccd)[terminos], 3),
  SE_CCD  = round(sqrt(diag(vcov(modelo_ccd)))[terminos], 3)
)
print(comp, row.names=FALSE)

## 5. Gráfico comparativo de superficies

In [ ]:
options(repr.plot.width=11, repr.plot.height=5)
m32  <- rsm(adherencia ~ SO(x1,x2), data=df32)
mccd <- rsm(adherencia ~ SO(x1,x2), data=df_ccd)
par(mfrow=c(1,2))
contour(m32,  ~x1+x2, image=TRUE, col.image=terrain.colors(20),
        main='3² (9 corridas)',    xlab='x1',ylab='x2')
contour(mccd, ~x1+x2, image=TRUE, col.image=terrain.colors(20),
        main='CCD (16 corridas)', xlab='x1',ylab='x2')

## 6. Conclusión

- El $3^2$ ajusta el modelo de 2° orden con solo **3 gl de error**: la estimación de
  $\sigma^2$ es poco estable y los SE de todos los coeficientes son altos.
- Añadir 4 puntos axiales ($\alpha=\sqrt{2}$) y 3 centros adicionales amplía los gl de
  error a **10**, mejorando la estimación de $\sigma^2$ y **reduciendo los SE**, especialmente
  para los coeficientes cuadráticos $\beta_{11}$ y $\beta_{22}$.
- **Nota:** un CCD rotable estándar parte solo de los 4 vértices ($2^2$); incluir los puntos
  de borde del $3^2$ produce un diseño aumentado no estándar — no es un CCD rotable clásico.
- El punto estacionario del modelo aumentado proporciona una buena estimación del óptimo para
  centrar el siguiente diseño confirmatorio.